In [ ]:
import pandas as pd
import numpy as np
import math
import random
import gc
from collections import defaultdict, Counter
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
import os
import gdown

In [ ]:
# 1. zip 파일의 ID 입력
zip_file_id = 'YOUR_DRIVE_FILE_ID'
output_filename = 'data.zip'

# 2. zip 파일 다운로드
url = f'https://drive.google.com/uc?id={zip_file_id}'
gdown.download(url, output_filename, quiet=False)

# 3. 압축 풀기 (리눅스 명령어를 사용합니다)
# -o: 덮어쓰기 허용, -d: 압축 풀 폴더 지정
!unzip -o data.zip

# 4. 확인
print("파일 목록:", os.listdir())
try:
    train = pd.read_csv('data/train.csv', encoding='utf-8-sig')
    test = pd.read_csv('data/test.csv', encoding='utf-8-sig')
    sample_submission = pd.read_csv('data/sample_submission.csv', encoding='utf-8-sig')
except FileNotFoundError:
    print("파일을 업로드해주세요.")
    exit()

In [ ]:
# ==========================================
# 1. 자모 유틸리티 & 정렬
# ==========================================
BASE_CODE = 0xAC00
CHOSUNG  = 21 * 28
JUNGSUNG = 28
CHOSUNG_LIST = ['ㄱ','ㄲ','ㄴ','ㄷ','ㄸ','ㄹ','ㅁ','ㅂ','ㅃ','ㅅ','ㅆ','ㅇ','ㅈ','ㅉ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']
JUNGSUNG_LIST = ['ㅏ','ㅐ','ㅑ','ㅒ','ㅓ','ㅔ','ㅕ','ㅖ','ㅗ','ㅘ','ㅙ','ㅚ','ㅛ','ㅜ','ㅝ','ㅞ','ㅟ','ㅠ','ㅡ','ㅢ','ㅣ']
JONGSUNG_LIST = ['', 'ㄱ','ㄲ','ㄳ','ㄴ','ㄵ','ㄶ','ㄷ','ㄹ','ㄺ','ㄻ','ㄼ','ㄽ','ㄾ','ㄿ','ㅀ','ㅁ','ㅂ','ㅄ','ㅅ','ㅆ','ㅇ','ㅈ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']

def to_jamo_list(text):
    result = []
    for ch in text:
        code = ord(ch) - BASE_CODE
        if code < 0 or code > (0xD7A3 - 0xAC00):
            result.append(ch)
        else:
            cho  = code // CHOSUNG
            jung = (code % CHOSUNG) // JUNGSUNG
            jong = (code % CHOSUNG) % JUNGSUNG
            result.extend([CHOSUNG_LIST[cho], JUNGSUNG_LIST[jung], JONGSUNG_LIST[jong]])
    return result

def from_jamo_list(jamo_list):
    result = []
    i = 0
    while i < len(jamo_list) - 2:
        cho, jung, jong = jamo_list[i], jamo_list[i+1], jamo_list[i+2]
        if cho in CHOSUNG_LIST and jung in JUNGSUNG_LIST and jong in JONGSUNG_LIST:
            cho_i = CHOSUNG_LIST.index(cho)
            jung_i = JUNGSUNG_LIST.index(jung)
            jong_i = JONGSUNG_LIST.index(jong)
            code = BASE_CODE + (cho_i * CHOSUNG) + (jung_i * JUNGSUNG) + jong_i
            result.append(chr(code))
            i += 3
        else:
            result.append(cho)
            i += 1
    return "".join(result)

def align_jamo_list(src_list, tgt_list):
    n, m = len(src_list), len(tgt_list)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(1, n+1): dp[i][0] = i
    for j in range(1, m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if src_list[i-1] == tgt_list[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    alignment = []
    i, j = n, m
    while i>0 or j>0:
        if i>0 and dp[i][j] == dp[i-1][j] + 1:
            alignment.append((src_list[i-1], '<DEL>')); i-=1
        elif j>0 and dp[i][j] == dp[i][j-1] + 1:
            alignment.append((None, tgt_list[j-1])); j-=1
        else:
            alignment.append((src_list[i-1], tgt_list[j-1])); i-=1; j-=1
    return alignment[::-1]

# ==========================================
# 2. [핵심 수정] 메모리 안전 데이터 생성기 (Chunk 방식)
# ==========================================
def create_dataset_generator(df, token_to_id, window_size=7, chunk_size=1000):
    """
    데이터를 한꺼번에 리스트에 담지 않고, 작은 덩어리(chunk)로 나누어 처리 후
    즉시 Numpy Array(int16)로 변환하여 메모리를 확보함.
    """
    data_X_list = []
    data_y_list = []
    
    PAD = '<PAD>'
    pad_id = token_to_id[PAD]
    
    # 임시 리스트 (chunk_size만큼만 담음)
    temp_X = []
    temp_y = []
    
    print(f"데이터셋 생성 중 (Chunking 적용)...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        in_jamos = to_jamo_list(row['input'])
        out_jamos = to_jamo_list(row['output'])
        alignment = align_jamo_list(in_jamos, out_jamos)
        
        # 윈도우 생성을 위한 정수 인코딩 미리 수행
        in_ids = [token_to_id.get(t, 0) for t in in_jamos]
        padded_ids = [pad_id] * (window_size // 2) + in_ids + [pad_id] * (window_size // 2)
        
        current_input_idx = 0
        
        for src_char, tgt_char in alignment:
            if src_char is None: continue 
            
            # 윈도우 추출 (이미 ID로 변환됨)
            window = padded_ids[current_input_idx : current_input_idx + window_size]
            target = token_to_id.get(tgt_char, 0)
            
            temp_X.append(window)
            temp_y.append(target)
            current_input_idx += 1
            
        # 청크 사이즈가 차면 Numpy로 압축하고 리스트 비우기
        if len(temp_X) >= chunk_size * 20: # 적절한 버퍼 크기
            data_X_list.append(np.array(temp_X, dtype=np.int16))
            data_y_list.append(np.array(temp_y, dtype=np.int16))
            temp_X = [] # 메모리 해제
            temp_y = []
            
    # 남은 데이터 처리
    if temp_X:
        data_X_list.append(np.array(temp_X, dtype=np.int16))
        data_y_list.append(np.array(temp_y, dtype=np.int16))
        
    print("청크 병합 중...")
    # 모든 청크를 하나로 합침
    X_final = np.concatenate(data_X_list, axis=0)
    y_final = np.concatenate(data_y_list, axis=0)
    
    # 사용한 리스트 즉시 삭제
    del data_X_list, data_y_list, temp_X, temp_y
    gc.collect()
    
    return X_final, y_final

# ==========================================
# 3. 언어 모델 (Language Model)
# ==========================================
def build_language_model(train_df):
    print("언어 모델(Bigram) 구축 중...")
    lm_counter = Counter()
    for txt in train_df['output']:
        jamos = [(None)] + to_jamo_list(txt)
        for i in range(len(jamos)-1):
            lm_counter[(jamos[i], jamos[i+1])] += 1
            
    lm_prob = defaultdict(lambda: 1e-7)
    total_counts = defaultdict(int)
    for (prev, curr), count in lm_counter.items():
        total_counts[prev] += count
    for (prev, curr), count in lm_counter.items():
        lm_prob[(prev, curr)] = count / total_counts[prev]
    return lm_prob

# ==========================================
# 4. 하이브리드 디코더 (메모리 관리 포함)
# ==========================================
def decode_hybrid(text, model, token_to_id, id_to_token, classes, lm_prob, vocab_set, 
                  window_size=7, beam_size=10, lm_weight=2.5, threshold=3.0):
    
    in_jamos = to_jamo_list(text)
    PAD = '<PAD>'
    pad_id = token_to_id.get(PAD, 0)
    
    # 윈도우 생성 (ID로 바로 변환)
    in_ids = [token_to_id.get(t, 0) for t in in_jamos]
    padded_ids = [pad_id] * (window_size // 2) + in_ids + [pad_id] * (window_size // 2)
    
    X_enc = []
    for i in range(len(in_jamos)):
        X_enc.append(padded_ids[i : i + window_size])
    
    # Numpy 변환 (int16)
    X_enc = np.array(X_enc, dtype=np.int16)
    
    # 예측
    probs = model.predict_proba(X_enc)
    
    beam = [([], None, 0.0)] 
    
    for i in range(len(in_jamos)):
        curr_probs = probs[i]
        top_k_indices = np.argsort(curr_probs)[-beam_size:] 
        
        new_beam = []
        for (decoded, last_char, score) in beam:
            for idx in top_k_indices:
                char_id = classes[idx]
                char_prob = curr_probs[idx]
                pred_char = id_to_token[char_id]
                
                ml_score = math.log(char_prob + 1e-9)
                lm_p = lm_prob.get((last_char, pred_char), 1e-7)
                lm_score = math.log(lm_p)
                
                total_score = score + ml_score + (lm_weight * lm_score)
                
                if pred_char == '<DEL>':
                    new_beam.append((decoded, last_char, total_score))
                else:
                    new_beam.append((decoded + [pred_char], pred_char, total_score))
        
        new_beam.sort(key=lambda x: x[2], reverse=True)
        beam = new_beam[:beam_size]
    
    candidates = []
    for (decoded, _, sc) in beam:
        word = from_jamo_list(decoded)
        candidates.append((word, sc))
        
    if not candidates: return text
    
    best_word, best_score = candidates[0]
    if best_word in vocab_set: return best_word
    
    for i in range(1, len(candidates)):
        cand_word, cand_score = candidates[i]
        if best_score - cand_score > threshold: break
        if cand_word in vocab_set: return cand_word
            
    return best_word

In [ ]:
# ==========================================
# 5. 메인 실행 프로세스 (완벽한 메모리 방어)
# ==========================================
# [1] 토큰 맵 미리 구축 (Chunking을 위해 필수)
print("토큰 맵 구축 중...")
all_text = "".join(train['input'].tolist()) + "".join(train['output'].tolist())
all_jamos = set(to_jamo_list(all_text)) | {'<PAD>', '<DEL>'}
token_to_id = {t: i for i, t in enumerate(sorted(list(all_jamos)))}
id_to_token = {i: t for t, i in token_to_id.items()}

# [2] 데이터셋 생성 (Chunk 방식 - 메모리 터짐 방지)
X_train, y_train = create_dataset_generator(train, token_to_id, window_size=7)

print(f"학습 데이터: {X_train.shape} (dtype={X_train.dtype})")
gc.collect() # 중요: 찌꺼기 메모리 청소

# [3] RandomForest 학습
print("RandomForest 학습 시작 (n_jobs=1)...")
rf_model = RandomForestClassifier(
    n_estimators=100, 
    n_jobs=1,        # 1로 고정 (필수)
    max_depth=20,    
    random_state=42, 
    verbose=1
)
rf_model.fit(X_train, y_train)

# 학습 데이터 메모리 해제 (이제 필요 없음)
del X_train, y_train
gc.collect()
print("학습 완료 & 메모리 청소")

# [4] LM & 사전 구축
lm_prob = build_language_model(train)
vocab_set = set()
for txt in train['output']: 
    for w in txt.split(): vocab_set.add(w)

# [5] 추론
print("하이브리드 추론 시작...")
outputs = []
rf_classes = rf_model.classes_

for idx, row in tqdm(test.iterrows(), total=len(test)):
    obf_text = row['input']
    tokens = obf_text.split()
    new_tokens = []
    
    for t in tokens:
        restored = decode_hybrid(t, rf_model, token_to_id, id_to_token, rf_classes, lm_prob, vocab_set,
                                 window_size=7, beam_size=10, lm_weight=2.5, threshold=3.0)
        new_tokens.append(restored)
    outputs.append(" ".join(new_tokens))
    
    if idx % 100 == 0:
        gc.collect()

In [ ]:
sample_submission['output'] = outputs
sample_submission.to_csv('submission_final_fixed.csv', index=False)
print("완료! submission_final_fixed.csv 저장됨.")